# L79 · 本地大模型推理 — KV-Cache 原理、top-k / top-p 采样、量化推理

**目标**：在本地运行量化的大语言模型，理解 KV-cache、温度参数与流式生成的工作机制。

🔗 **Aurora 连接**：本节是 `aurora.llm` 子包的基础；Podcast Agent（`aurora.agents.podcast`）的语言模型后端直接依赖这里建立的本地推理管道。无 API、无网络、可离线运行。

## 核心直觉

量化把浮点权重"截断"成低精度整数，让 7B 参数模型从 14 GB 压缩到 4 GB 以内，普通消费级 GPU 或 Apple Silicon 即可装入。推理时 KV-cache 把每一步已算好的注意力中间值缓存起来，使自回归生成从 O(N²) 降到 O(N)。Temperature 则在 softmax 之前对 logits 做一次缩放，控制输出的随机程度——这三个机制合在一起，决定了本地 LLM 的速度、显存占用和生成质量。

In [ ]:
# 需要：pip install transformers bitsandbytes accelerate
# 推荐模型：Qwen2.5-7B-Instruct 或 Llama-3.1-8B-Instruct
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, time, textwrap

## 1. 4-bit / 8-bit 量化

FP16 每个权重占 2 字节；INT8 占 1 字节；INT4 仅占 0.5 字节。量化把连续浮点值映射到离散整数格，公式为：

```
x_quant = round(x / scale) + zero_point
x_dequant = (x_quant - zero_point) * scale
```

`scale` 和 `zero_point` 按 group（通常 128 个权重一组）存储为 FP16，推理时先反量化再做矩阵乘法。4-bit 量化的精度损失在困惑度（perplexity）上约 +0.5～1.0，对对话任务几乎无感知差异。`bitsandbytes` 库实现了 NF4（Normal Float 4）格式，对高斯分布权重的精度损失比均匀 INT4 更小。

In [ ]:
# 演示：量化 / 反量化一个小向量，观察误差
import numpy as np

np.random.seed(42)
weights = np.random.randn(16).astype(np.float32)  # 模拟一行权重

def quantize_int8(x):
    scale = x.max() / 127.0
    q = np.round(x / scale).astype(np.int8)
    dq = q.astype(np.float32) * scale
    return q, dq, scale

def quantize_int4(x):
    scale = x.max() / 7.0
    q = np.clip(np.round(x / scale), -8, 7).astype(np.int8)
    dq = q.astype(np.float32) * scale
    return q, dq, scale

_, dq8, s8 = quantize_int8(weights)
_, dq4, s4 = quantize_int4(weights)

err8 = np.abs(weights - dq8).mean()
err4 = np.abs(weights - dq4).mean()

print(f"原始权重（前8个）: {weights[:8].round(4)}")
print(f"INT8 反量化（前8）: {dq8[:8].round(4)}")
print(f"INT4 反量化（前8）: {dq4[:8].round(4)}")
print(f"INT8 平均绝对误差: {err8:.5f}")
print(f"INT4 平均绝对误差: {err4:.5f}")
print(f"内存比较 FP32:INT8:INT4 = 4B : 1B : 0.5B ≈ 8x : 2x : 1x")
assert err8 < err4, "INT8 误差应小于 INT4"
print("✅ 量化误差符合预期")

## 2. KV-cache

Transformer 的自回归推理每次生成一个 token，需要对所有历史 token 重新计算注意力（attention）。若上下文长度为 N，每步计算量为 O(N)，N 步合计 O(N²)。

KV-cache 的做法：将每一层的 Key 矩阵和 Value 矩阵按时间步缓存：

```
# 第 t 步
K_cache[:, :t, :] = 历史 key
V_cache[:, :t, :] = 历史 value
# 只需计算当前 token 的 q，与完整 K_cache 做点积
attn = softmax(q @ K_cache.T / sqrt(d_k)) @ V_cache
```

每步只需追加一行到 cache，计算量降为 O(1) per step，总计 O(N)。代价是显存：对于 7B 模型、32 层、2048 token 上下文，KV-cache 约占 `32 * 2 * 2048 * 128 * 32 * 2 bytes ≈ 1 GB`（FP16）。

In [ ]:
# 演示：对比有无 KV-cache 的计算步数（用矩阵乘法次数模拟）
import numpy as np

def attn_no_cache(tokens):
    """每步重新算所有 token 的 K, V"""
    total_ops = 0
    for t in range(1, len(tokens) + 1):
        total_ops += t * t  # Q(1) × K(t) 是 t 次乘法
    return total_ops

def attn_with_cache(tokens):
    """每步只算当前 token，追加到 cache"""
    total_ops = 0
    for t in range(1, len(tokens) + 1):
        total_ops += t  # Q(1) × K_cache(t)：t 次乘法（但仅新增 1 行）
    return total_ops

ns = [64, 128, 256, 512, 1024]
print(f"{'上下文长度':>10} {'无cache(O(N²))':>18} {'有cache(O(N))':>15} {'加速比':>8}")
for n in ns:
    tokens = list(range(n))
    no_c = attn_no_cache(tokens)
    c    = attn_with_cache(tokens)
    print(f"{n:>10,} {no_c:>18,} {c:>15,} {no_c/c:>8.1f}x")

assert attn_no_cache([0]*512) > attn_with_cache([0]*512) * 100
print("✅ KV-cache 在长上下文下显著减少计算量")

## 3. Temperature（温度参数）

LLM 每步输出一个 logits 向量（维度 = 词表大小，约 32000）。Temperature T 在 softmax 之前对 logits 做缩放：

```
probs = softmax(logits / T)
```

- `T → 0`（如 0.01）：最大 logit 被无限放大，softmax 退化为 argmax，  输出完全确定性（贪婪解码）。
- `T = 1.0`：原始分布，模型训练时的默认行为。
- `T > 1`（如 1.5）：logits 差异缩小，分布变"平"，低概率词被采样的机会增大，  输出更随机、更多样，但可能失去连贯性。

实践中对话任务常用 `T = 0.7`；创意写作用 `T = 0.9~1.1`；代码生成用 `T = 0.1~0.3`（需确定性）。

In [ ]:
import numpy as np

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

# 模拟一个小词表（5个词）的 logits
np.random.seed(7)
logits = np.array([3.2, 1.5, 0.8, 2.7, 0.1])
words  = ["音乐", "风格", "节奏", "旋律", "噪声"]

print(f"{'词':>6} {'logit':>8} ", end="")
for T in [0.1, 0.7, 1.5]:
    print(f"  T={T}", end="")
print()

for i, (w, l) in enumerate(zip(words, logits)):
    print(f"{w:>6} {l:>8.2f} ", end="")
    for T in [0.1, 0.7, 1.5]:
        p = softmax(logits / T)
        print(f"  {p[i]:.3f}", end="")
    print()

print()
for T in [0.1, 0.7, 1.5]:
    p = softmax(logits / T)
    sampled = np.random.choice(words, p=p, size=10)
    unique = len(set(sampled))
    print(f"T={T}: 10次采样唯一词数={unique}, 分布熵={(-p*np.log(p+1e-12)).sum():.3f}")

assert softmax(logits / 0.01).max() > 0.99, "低温应接近 one-hot"
assert (-softmax(logits / 0.01) * np.log(softmax(logits / 0.01) + 1e-12)).sum() <        (-softmax(logits / 1.5)  * np.log(softmax(logits / 1.5)  + 1e-12)).sum(),        "高温熵应更大"
print("✅ Temperature 对分布的影响符合预期")

## 参数实验：同一提示词，三种 Temperature 各生成 5 次

**目标**：直观感受 `temperature` 对多样性与连贯性的影响。

| 参数 | 值 | 预期现象 |
|------|----|---------|
| `temperature=0.1` | 接近贪婪 | 5次输出几乎相同，措辞固定，逻辑清晰 |
| `temperature=0.7` | 平衡点 | 内容相近但措辞有变化，连贯性仍好 |
| `temperature=1.5` | 高随机 | 每次输出差异大，偶尔出现不连贯或重复 |

运行前须先加载模型（见下方 code 格）。若显存不足，改用 `load_in_4bit=True` 量化加载。

In [ ]:
# ── 加载模型（首次运行会下载权重，约 4-14 GB）──────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # 或 "meta-llama/Llama-3.1-8B-Instruct"

# 4-bit 量化（显存 < 8 GB 时推荐）
quantization_config = None
try:
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    print("已启用 NF4 4-bit 量化")
except Exception as e:
    print(f"bitsandbytes 不可用，使用 FP16: {e}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.eval()
print(f"模型加载完毕：{MODEL_ID}")

In [ ]:
# ── Temperature 对比实验 ──────────────────────────────────────────────────────
PROMPT = "用一句话描述爵士乐的核心特征："
TEMPERATURES = [0.1, 0.7, 1.5]
N_RUNS = 5

messages = [{"role": "user", "content": PROMPT}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

for T in TEMPERATURES:
    print(f"\n{'='*60}")
    print(f"temperature = {T}  （{N_RUNS} 次生成）")
    print('='*60)
    outputs_set = set()
    for i in range(N_RUNS):
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=60,
                temperature=T,
                do_sample=(T > 0.05),   # T≈0 时用贪婪
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(
            out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
        ).strip()
        outputs_set.add(generated)
        print(f"  [{i+1}] {textwrap.shorten(generated, width=80)}")
    print(f"  → 唯一输出数：{len(outputs_set)} / {N_RUNS}")

## 本课收束

本节通过 `AutoModelForCausalLM.generate()` 在本地运行量化 LLM，输出是 token id 序列，经 `tokenizer.decode()` 还原为文本。这些能力封装进 `aurora.llm.local_engine`，为 Podcast Agent 提供零 API 依赖的语言后端。下一节（M5-L4）将基于此引擎实现流式生成（streaming）与结构化输出（JSON mode），让 Agent 能实时吐字并解析可靠的 JSON 指令。